#### Cell 1 — Imports and source

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_date, lit

df_source = spark.table("silver_customers")

#### Cell 2 — Create the dimension only if it doesn't exist

In [0]:
if not spark.catalog.tableExists("dim_customer_scd2"):

    df_initial = (
        df_source
        .withColumn("effective_start_date", current_date())
        .withColumn("effective_end_date", lit(None).cast("date"))
        .withColumn("is_current", lit(True))
    )

    df_initial.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("dim_customer_scd2")

#### Cell 3 — Detect changed customers

In [0]:
df_current = (
    spark.table("dim_customer_scd2")
    .filter(col("is_current") == True)
)

df_changes = (
    df_source.alias("source")
    .join(
        df_current.alias("target"),
        "customer_id",
        "inner"
    )
    .filter(
        (col("source.name") != col("target.name")) |
        (col("source.city") != col("target.city")) |
        (col("source.state") != col("target.state"))
    )
    .select("source.*")
)

#### Cell 4 — Close changed records

In [0]:
customer_dim = DeltaTable.forName(spark, "dim_customer_scd2")

(
    customer_dim.alias("target")
    .merge(
        df_changes.alias("source"),
        """
        target.customer_id = source.customer_id
        AND target.is_current = true
        """
    )
    .whenMatchedUpdate(
        set={
            "effective_end_date": current_date(),
            "is_current": lit(False)
        }
    )
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

#### Cell 5 — Insert the new versions

In [0]:
df_new_versions = (
    df_changes
    .withColumn("effective_start_date", current_date())
    .withColumn("effective_end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
)

if not df_new_versions.isEmpty():
    df_new_versions.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("dim_customer_scd2")